# 🔄 LlamaIndex — Ingestion Pipeline সম্পূর্ণ গাইড

## Ingestion Pipeline কী?

আগে আমরা যেভাবে কাজ করতাম:
```
❌ পুরনো পদ্ধতি (Manual):
documents = reader.load_data()       # Step 1
nodes = splitter.get_nodes(docs)     # Step 2  
index = VectorStoreIndex(nodes)      # Step 3 (embedding এখানে হয়)
```

**Ingestion Pipeline** দিয়ে সব একসাথে:
```
✅ নতুন পদ্ধতি (Pipeline):
pipeline = IngestionPipeline(
    transformations=[splitter, embed_model],
    vector_store=my_vector_store
)
pipeline.run(documents=documents)   # সব একসাথে!
```

## কেন ব্যবহার করব?
| সুবিধা | বিবরণ |
|--------|-------|
| **Cache** | একই document দুইবার process হয় না (সময় বাঁচে) |
| **Clean Code** | সব transformation এক জায়গায় |
| **Parallel** | একসাথে অনেক document process করা যায় |
| **Vector Store সরাসরি** | process করে সরাসরি Chroma/Pinecone-এ save |

## এই নোটবুকে যা শিখবে:
1. ✅ Basic Ingestion Pipeline
2. ✅ Pipeline Cache (Duplicate Skip)
3. ✅ ChromaDB-এর সাথে Pipeline
4. ✅ Pipeline Disk-এ Save/Load
5. ✅ Custom Transformation তৈরি করা
6. ✅ পুরো Production Pipeline

---
## ⚙️ Step 0: Setup

In [ ]:
# ─── প্রয়োজনীয় লাইব্রেরি ───
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import MockLLM

# Embedding Model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# LLM (টেস্টের জন্য MockLLM, real-এ OpenAI দাও)
Settings.llm = MockLLM(max_tokens=256)

# OpenAI দিয়ে করতে চাইলে:
# from dotenv import load_dotenv
# from llama_index.llms.openai import OpenAI
# load_dotenv()
# Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

print("✅ Setup সম্পন্ন!")

---
## 📌 Step 1: Basic Ingestion Pipeline
সবচেয়ে সহজ Pipeline — Documents লোড করে split করে।

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

# ── Documents লোড করা ──
documents = SimpleDirectoryReader(input_dir="Data").load_data()
print(f"📄 {len(documents)}টি document লোড হয়েছে")
for doc in documents:
    print(f"   → {doc.metadata.get('file_name', '?')}")

# ── Pipeline তৈরি করা ──
pipeline = IngestionPipeline(
    transformations=[
        # Transformation 1: Split করা
        SentenceSplitter(
            chunk_size=256,
            chunk_overlap=50
        ),
        # Transformation 2: Embedding করা
        # (Settings.embed_model এখানে automatically ব্যবহার হবে)
        Settings.embed_model,
    ]
)

# ── Pipeline চালানো ──
nodes = pipeline.run(documents=documents)

print(f"\n✅ Pipeline সম্পন্ন!")
print(f"📦 মোট {len(nodes)}টি Node তৈরি হয়েছে")

In [ ]:
# ── তৈরি হওয়া Nodes দেখা ──
print("📋 প্রথম ৩টি Node-এর বিবরণ:\n")

for i, node in enumerate(nodes[:3]):
    print(f"{'━'*55}")
    print(f"🔹 Node {i+1}")
    print(f"   ID: {node.node_id[:25]}...")
    print(f"   File: {node.metadata.get('file_name', 'N/A')}")
    print(f"   Text (প্রথম ৮০ char): {node.text[:80]}...")
    
    # Embedding আছে কিনা দেখা
    if node.embedding:
        print(f"   Embedding Size: {len(node.embedding)} dimensions")
    else:
        print(f"   Embedding: None (embed_model দাওনি)")

---
## 📌 Step 2: Pipeline Cache — একই Document দুইবার Process হবে না!
এটাই Ingestion Pipeline-এর সবচেয়ে বড় সুবিধা।

> **সমস্যা:** ধরো তোমার কাছে ১০০টি file আছে। প্রতিদিন ১-২টা নতুন file যোগ হয়।  
> Cache না থাকলে: প্রতিদিন ১০০টা file-ই আবার process করতে হবে। 😢  
> Cache থাকলে: শুধু নতুন ১-২টা file process হবে! 😊

In [ ]:
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.node_parser import SentenceSplitter
import time

# ── Cache সহ Pipeline ──
pipeline_with_cache = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),
        Settings.embed_model,
    ],
    cache=IngestionCache()   # ← এখানে Cache যোগ করা হলো
)

# ── প্রথমবার চালানো ──
print("▶️  প্রথমবার Pipeline চালানো হচ্ছে...")
start = time.time()
nodes1 = pipeline_with_cache.run(documents=documents)
first_run_time = time.time() - start
print(f"   ✅ {len(nodes1)}টি Node তৈরি। সময় লেগেছে: {first_run_time:.2f}s")

print()

# ── দ্বিতীয়বার একই documents দিয়ে চালানো ──
print("▶️  দ্বিতীয়বার একই documents দিয়ে চালানো হচ্ছে...")
start = time.time()
nodes2 = pipeline_with_cache.run(documents=documents)
second_run_time = time.time() - start
print(f"   ✅ {len(nodes2)}টি Node তৈরি। সময় লেগেছে: {second_run_time:.2f}s")

print(f"\n⚡ দ্বিতীয়বার অনেক দ্রুত হয়েছে! Cache এর কারণে।")
print(f"   প্রথমবার: {first_run_time:.2f}s")
print(f"   দ্বিতীয়বার: {second_run_time:.2f}s (cache hit!)")

---
## 📌 Step 3: ChromaDB-এর সাথে Ingestion Pipeline
Process করা Nodes সরাসরি ChromaDB-তে save করা।

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.core import StorageContext
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore

# ── ChromaDB Setup ──
chroma_client = chromadb.Client()   # In-memory
# Persistent করতে চাইলে:
# chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="pipeline_collection"
)

vector_store = ChromaVectorStore(chroma_collection=collection)

print(f"✅ ChromaDB Collection তৈরি: '{collection.name}'")

# ── Vector Store সহ Pipeline ──
pipeline_chroma = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),
        Settings.embed_model,
    ],
    vector_store=vector_store   # ← সরাসরি Chroma-তে save হবে!
)

# ── Pipeline চালানো ──
print("\n🔄 Pipeline চালানো হচ্ছে (Chroma-তে save হবে)...")
nodes = pipeline_chroma.run(documents=documents)

print(f"✅ {len(nodes)}টি node process হয়েছে!")
print(f"📦 Chroma-তে মোট items: {collection.count()}")

In [ ]:
# ── Chroma থেকে Index বানিয়ে Query করা ──
storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# Index তৈরি (Nodes ইতোমধ্যে Chroma-তে আছে, নতুন করে process হবে না)
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_context
)

query_engine = index.as_query_engine(similarity_top_k=2)

response = query_engine.query("What is LlamaIndex?")
print("🤖 উত্তর:")
print(response)
print(f"\n📂 Source: {[n.metadata.get('file_name','?') for n in response.source_nodes]}")

---
## 📌 Step 4: Pipeline ও Cache Disk-এ Save ও Load করা
Pipeline-কে সেভ করলে পরের বার cache-ও সাথে থাকে।

In [ ]:
import os
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.storage.docstore import SimpleDocumentStore

PIPELINE_DIR = "./pipeline_storage"
os.makedirs(PIPELINE_DIR, exist_ok=True)

# ── Cache সহ Pipeline ──
persistent_pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),
        Settings.embed_model,
    ],
    cache=IngestionCache(),
    docstore=SimpleDocumentStore()   # Document-এর hash track করে duplicate এড়ায়
)

# Pipeline চালানো
nodes = persistent_pipeline.run(documents=documents)
print(f"✅ {len(nodes)}টি node তৈরি!")

# ── Pipeline Disk-এ Save ──
persistent_pipeline.persist(persist_dir=PIPELINE_DIR)
print(f"\n💾 Pipeline '{PIPELINE_DIR}' ফোল্ডারে save হয়েছে!")
print("📁 Save হওয়া ফাইলগুলো:")
for f in os.listdir(PIPELINE_DIR):
    print(f"   → {f}")

In [ ]:
# ── Save করা Pipeline Load করা ──
loaded_pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),
        Settings.embed_model,
    ]
)

# Disk থেকে cache load
loaded_pipeline.load(persist_dir=PIPELINE_DIR)
print("✅ Pipeline load সম্পন্ন!")

# একই documents আবার দিলে cache hit হবে (দ্রুত হবে)
import time
start = time.time()
nodes_cached = loaded_pipeline.run(documents=documents)
elapsed = time.time() - start
print(f"⚡ Cache থেকে load: {len(nodes_cached)}টি node, সময়: {elapsed:.2f}s")

---
## 📌 Step 5: Custom Transformation তৈরি করা
নিজের মতো transformation বানানো যায়!  
যেমন: automatically metadata যোগ করা, text clean করা, etc.

In [ ]:
from llama_index.core.schema import TransformComponent, BaseNode
from typing import Any, List

# ════════════════════════════════════════════════════
#  Custom Transformation 1: Metadata Enricher
#  → প্রতিটা Node-এ extra metadata যোগ করবে
# ════════════════════════════════════════════════════
class MetadataEnricher(TransformComponent):
    """প্রতিটা Node-এ custom metadata যোগ করে"""
    
    project_name: str = "LlamaIndex_Practice"
    author: str = "Maruf"
    
    def __call__(self, nodes: List[BaseNode], **kwargs: Any) -> List[BaseNode]:
        for node in nodes:
            # Extra metadata যোগ করা
            node.metadata["project"]   = self.project_name
            node.metadata["author"]    = self.author
            node.metadata["processed"] = "True"
            node.metadata["char_count"] = str(len(node.text))
        return nodes


# ════════════════════════════════════════════════════
#  Custom Transformation 2: Text Cleaner
#  → Text থেকে extra whitespace, newline পরিষ্কার করে
# ════════════════════════════════════════════════════
class TextCleaner(TransformComponent):
    """Node-এর text পরিষ্কার করে"""
    
    def __call__(self, nodes: List[BaseNode], **kwargs: Any) -> List[BaseNode]:
        for node in nodes:
            # Multiple whitespace → single space
            import re
            cleaned = re.sub(r'\s+', ' ', node.text).strip()
            node.text = cleaned
        return nodes


print("✅ Custom Transformations তৈরি হয়েছে:")
print("   → MetadataEnricher")
print("   → TextCleaner")

In [ ]:
# ── Custom Transformation-সহ Pipeline ──
custom_pipeline = IngestionPipeline(
    transformations=[
        TextCleaner(),                                    # Step 1: Text পরিষ্কার করা
        SentenceSplitter(chunk_size=256, chunk_overlap=50), # Step 2: Split করা
        MetadataEnricher(project_name="RAG_Demo", author="Maruf"),  # Step 3: Metadata যোগ
        Settings.embed_model,                             # Step 4: Embedding
    ]
)

custom_nodes = custom_pipeline.run(documents=documents)

# ── Custom Metadata দেখা ──
print(f"✅ {len(custom_nodes)}টি Node তৈরি!\n")
print("📋 প্রথম Node-এর Metadata (custom field সহ):")
for key, val in custom_nodes[0].metadata.items():
    print(f"   {key:20} : {val}")

---
## 📌 Step 6: Docstore — Duplicate Document এড়ানো
`docstore` ব্যবহার করলে নতুন document detect হয় এবং পুরনো document skip হয়।

> **Real-world scenario:** প্রতিদিন নতুন ফাইল যোগ হচ্ছে।  
> Docstore মনে রাখে কোন ফাইল আগে দেখেছে, কোনটা নতুন।


In [ ]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.schema import Document

# ── Docstore সহ Pipeline ──
docstore = SimpleDocumentStore()

docstore_pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),
        Settings.embed_model,
    ],
    docstore=docstore   # ← এখানে docstore যোগ
)

# ── প্রথমবার — সব documents process হবে ──
print("▶️  প্রথমবার: Original documents")
nodes1 = docstore_pipeline.run(documents=documents)
print(f"   ✅ {len(nodes1)}টি node তৈরি")

# ── নতুন document যোগ করা ──
new_doc = Document(
    text="This is a brand new document about Python programming.",
    metadata={"file_name": "new_python.txt", "source": "manual"}
)
new_documents = documents + [new_doc]  # পুরনো + নতুন

# ── দ্বিতীয়বার — শুধু নতুন document process হবে ──
print("\n▶️  দ্বিতীয়বার: পুরনো + ১টি নতুন document")
nodes2 = docstore_pipeline.run(documents=new_documents)
print(f"   ✅ {len(nodes2)}টি node তৈরি (শুধু নতুনটার জন্য)")
print("   💡 পুরনো documents skip হয়েছে!")

---
## 📌 Step 7: Parallel Processing — একসাথে অনেক Document
বড় dataset-এর জন্য parallel processing চালু করা।

In [ ]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.schema import Document
import time

# ── Test-এর জন্য অনেক document তৈরি ──
many_docs = [
    Document(
        text=f"এটি {i+1} নম্বর document। LlamaIndex RAG system তৈরিতে সাহায্য করে।",
        metadata={"doc_num": str(i+1)}
    )
    for i in range(20)  # ২০টি document
]

# ── Normal (Sequential) Pipeline ──
normal_pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=128, chunk_overlap=20),
        Settings.embed_model,
    ]
)

print("▶️  Normal (Sequential) Pipeline চালানো...")
start = time.time()
nodes_normal = normal_pipeline.run(documents=many_docs)
normal_time = time.time() - start
print(f"   ✅ {len(nodes_normal)} nodes, সময়: {normal_time:.2f}s")

print()

# ── Parallel Pipeline ──
parallel_pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=128, chunk_overlap=20),
        Settings.embed_model,
    ]
)

print("▶️  Parallel Pipeline চালানো... (num_workers=4)")
start = time.time()
nodes_parallel = parallel_pipeline.run(
    documents=many_docs,
    num_workers=4   # ← 4টি worker একসাথে কাজ করবে
)
parallel_time = time.time() - start
print(f"   ✅ {len(nodes_parallel)} nodes, সময়: {parallel_time:.2f}s")

---
## 📌 Step 8: পুরো Production-Ready Pipeline
Real project-এ যেভাবে ব্যবহার করা হয় — সব কিছু একসাথে।

**Features:**
- ✅ ChromaDB Persistent storage
- ✅ Cache (speed boost)
- ✅ Docstore (duplicate skip)
- ✅ Custom Transformation
- ✅ Auto-detect নতুন documents

In [ ]:
import os
import chromadb
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import MockLLM

# ══════════════════════════════════════════
#  ⚙️  CONFIGURATION
# ══════════════════════════════════════════
DATA_DIR      = "Data"
CHROMA_DIR    = "./chroma_prod"
PIPELINE_DIR  = "./pipeline_prod"
COLLECTION    = "production_collection"

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
Settings.llm = MockLLM(max_tokens=256)

# ══════════════════════════════════════════
#  🗄️  CHROMA SETUP (Persistent)
# ══════════════════════════════════════════
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection    = chroma_client.get_or_create_collection(COLLECTION)
vector_store  = ChromaVectorStore(chroma_collection=collection)
print(f"✅ ChromaDB ready. Existing items: {collection.count()}")

# ══════════════════════════════════════════
#  🔄  PIPELINE SETUP
# ══════════════════════════════════════════
pipeline = IngestionPipeline(
    transformations=[
        TextCleaner(),                                       # Custom: text পরিষ্কার
        SentenceSplitter(chunk_size=256, chunk_overlap=50),  # Split
        MetadataEnricher(project_name="Production_RAG"),     # Custom: metadata
        Settings.embed_model,                                # Embed
    ],
    vector_store = vector_store,   # Chroma-তে save
    cache        = IngestionCache(),
    docstore     = SimpleDocumentStore(),
)

# আগের cache load করা (যদি থাকে)
if os.path.exists(PIPELINE_DIR):
    print("📂 আগের Pipeline cache পাওয়া গেছে। Load হচ্ছে...")
    pipeline.load(persist_dir=PIPELINE_DIR)

# ══════════════════════════════════════════
#  📄  DOCUMENTS LOAD & PROCESS
# ══════════════════════════════════════════
print("\n📄 Documents লোড হচ্ছে...")
documents = SimpleDirectoryReader(input_dir=DATA_DIR).load_data()
print(f"   {len(documents)}টি document পাওয়া গেছে")

print("\n🔄 Pipeline চালানো হচ্ছে...")
import time
start = time.time()
nodes = pipeline.run(documents=documents)
elapsed = time.time() - start

print(f"✅ সম্পন্ন! {len(nodes)}টি node process হয়েছে ({elapsed:.2f}s)")
print(f"📦 Chroma-তে মোট items: {collection.count()}")

# Pipeline save করা
pipeline.persist(persist_dir=PIPELINE_DIR)
print(f"💾 Pipeline cache saved to '{PIPELINE_DIR}'")

# ══════════════════════════════════════════
#  🔍  QUERY ENGINE
# ══════════════════════════════════════════
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_context
)

query_engine = index.as_query_engine(similarity_top_k=3)

print("\n" + "═"*55)
questions = ["What is LlamaIndex?", "What is RAG?"]
for q in questions:
    print(f"\n❓ {q}")
    res = query_engine.query(q)
    print(f"🤖 {res.response}")
    srcs = [n.metadata.get('file_name','?') for n in res.source_nodes]
    print(f"📂 Sources: {srcs}")

---
## 🎯 সারসংক্ষেপ — যা শিখলে

| বিষয় | কোড |
|-------|-----|
| Basic Pipeline | `IngestionPipeline(transformations=[...])` |
| Pipeline চালানো | `pipeline.run(documents=docs)` |
| Cache যোগ | `cache=IngestionCache()` |
| Docstore যোগ | `docstore=SimpleDocumentStore()` |
| Chroma সাথে | `vector_store=ChromaVectorStore(...)` |
| Pipeline Save | `pipeline.persist(persist_dir="./dir")` |
| Pipeline Load | `pipeline.load(persist_dir="./dir")` |
| Custom Transformation | `class MyTransform(TransformComponent): ...` |
| Parallel | `pipeline.run(docs, num_workers=4)` |

## পুরনো vs নতুন পদ্ধতির তুলনা

```python
# ❌ পুরনো পদ্ধতি (Manual, কঠিন)
docs  = reader.load_data()
nodes = splitter.get_nodes_from_documents(docs)
for node in nodes:
    node.embedding = embed_model.get_text_embedding(node.text)
index = VectorStoreIndex(nodes, storage_context=storage_ctx)

# ✅ নতুন পদ্ধতি (Pipeline, সহজ)
pipeline = IngestionPipeline(
    transformations=[splitter, embed_model],
    vector_store=vector_store,
    cache=IngestionCache(),
    docstore=SimpleDocumentStore()
)
pipeline.run(documents=docs)  # শুধু এটুকু!
```

## পরবর্তী শেখার বিষয়
- 📌 **Evaluation** — RAG কতটা ভালো কাজ করছে পরিমাপ করা
- 📌 **Agents** — Tool ব্যবহার করে স্বয়ংক্রিয়ভাবে কাজ করা
- 📌 **Advanced Retrieval** — BM25, Reranking, Hybrid Search